# Scale extension — does convergence strengthen with encoder capacity?

**Companion to `convergence_figures.ipynb`. This one is an EXPERIMENT, not a figure script.**

## What this tests, and why

Huh et al.'s Platonic Representation Hypothesis plots alignment against language-model capability up to 70B parameters and finds it rises. This project measured convergence at a much smaller scale — text encoders at bge-m3/BERT/GPT-2 size (768–1024-d, ~0.1–0.6B params) — which is the single most likely objection at the defense (see `Defense_Brief` §8, `Mock_Viva` Tier 2, `Project_Atlas` "Scale the text side").

This notebook tests the prediction directly, on **both** sides:

- **Image side:** add **DINOv2-giant** (1.1B params) to the existing small→base→large ladder. The project already measured +15.3 points of ceiling per decade of parameters (R² 0.949) across three rungs. A fourth rung tests whether that line holds or bends.
- **Text side:** add one or two larger text encoders. This is the axis PRH actually plots and the one the project never varied.

Two quantities are measured for each new encoder:

1. **Raw shape agreement (ρ)** — Spearman of pairwise-distance structure against every existing encoder. No map fitted, nothing trained. This is the PRH-style alignment measure.
2. **Hub transfer (% of native)** — fit one entry map into the **existing frozen hub**, apply the **existing frozen caption head**, score R@1 against a natively-fitted head. This is the project's own measure, and it is genuinely zero-shot for the head.

## Why both, and not just one

Section E.12 established that these two do **not** track each other: ConvNeXt has the lowest raw agreement of any image encoder (ρ = 0.330) yet the highest transfer (96.7%). So "does scale help?" has two possible answers, and they can differ. Measuring only one would be the same mistake E.6 made on the text side.

## Cost, honestly

- DINOv2-giant over 9,533 images: ~15–25 min on an L4.
- A 4B text encoder over 47,665 captions (5 per image, averaged as in the original protocol): ~45–70 min.
- New caches: roughly 100–350 MB each. Check Drive space first.
- Everything downstream (maps, hub entry, transfer) is closed-form and takes seconds — consistent with the project's "one GPU pass, then a ruler" claim.

## Run order

Cells 1–2 configure and **pre-register**. Cell 3 loads existing artefacts. Cell 4 is the one you must edit — your COCO source. Cells 5–6 encode. Cells 7–9 measure. Cell 10 scores the result against the prediction written in Cell 2.

Stages cache to Drive and skip if already done, so the notebook is resumable.

In [ ]:
# Cell 1 — mount, install, configure
from google.colab import drive
drive.mount('/content/drive')

!pip -q install transformers timm sentence-transformers --upgrade 2>&1 | tail -1

import os, json, numpy as np, torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

FOLDER   = "/content/drive/MyDrive/convergence_experiment"
OUT      = os.path.join(FOLDER, "scale_extension")
os.makedirs(OUT, exist_ok=True)
DEV      = "cuda" if torch.cuda.is_available() else "cpu"

# ── which models to add ──────────────────────────────────────────────
# Image side: one more rung on the existing DINOv2 ladder.
#   The project uses cls+patch concatenation, so output dim = 2 x native.
#   small 384->768, base 768->1536, large 1024->2048, giant 1536->3072.
# POOLING drives everything: E1.1 sets corpus size by encoder width via
#   N_PAIRS = N_EVAL + 10 * width / 0.9   (~11 rows per input dimension)
# and the hub then uses 9,533 rows, fixed by the narrowest source (small, 768d).
# An entry map fitted on 8,533 train rows therefore gets 8533/width rows/dim.
# Below ~5 the ridge fit turns SYSTEMATICALLY PESSIMISTIC with no error raised -
# the sample-starvation bias documented in A1's review notes.
#
#   giant cls+patch = 3072d -> 2.8 rows/dim  (transfer fit underpowered)
#   giant cls       = 1536d -> 5.6 rows/dim  (transfer fit powered)
#
# BUT that constraint governs the TRANSFER fit only. rho fits nothing and has
# no rows/dim requirement at all. small/base/large are all cls+patch, so
# running giant on cls-only would confound capacity with pooling on exactly
# the axis being measured - and an earlier run of this notebook did that,
# producing a cross-modal rho of 0.238 for giant that cannot be interpreted.
# Giant therefore uses cls+patch, matching the other three rungs; its
# transfer number is auto-flagged inconclusive by the power gate in Cell 8.
IMAGE_MODELS = {
    "img_giant": dict(hf="facebook/dinov2-giant", params_b=1.14,
                      native_dim=1536, pooling="cls+patch"),     # 3072d, matches the ladder
    # Optional, ONLY if a powered transfer number for giant is wanted.
    # Its rho is NOT comparable to the other rungs.
    # "img_giant_cls": dict(hf="facebook/dinov2-giant", params_b=1.14,
    #                       native_dim=1536, pooling="cls"),      # 1536d
}

# Text side: a capacity ladder. Start with the smaller one; add the larger
# only if the first completes and Drive has room.
# Text side. THE SCALE LADDER, and note which test each rung serves.
#
#   bge-m3 (already in the hub)  0.568B  1024d
#   Qwen3-Embedding-0.6B         0.600B  1024d   <- only 1.06x bge-m3.
#                                                   NOT a scale step: same size,
#                                                   different architecture/data.
#                                                   Kept as an architecture control.
#   Qwen3-Embedding-4B           4.0B    2560d   <- 7.0x bge-m3. THE scale test.
#
# The two measurements have DIFFERENT requirements:
#   * rho (shape agreement) fits NOTHING - no rows-per-dim constraint at all,
#     so any size is free.
#   * hub transfer fits an entry map - needs >= 5 rows/dim (8,533 / width).
#       0.6B  1024d -> 8.3  powered
#       4B    2560d -> 3.3  underpowered; its transfer is auto-flagged
#                            inconclusive by the power gate in Cell 8.
# So 4B is included for rho (where it is the real scale test) and its
# transfer number is reported but not interpreted.
TEXT_MODELS = {
    "txt_qwen06": dict(hf="Qwen/Qwen3-Embedding-0.6B", params_b=0.6, role="architecture control"),
    "txt_qwen4":  dict(hf="Qwen/Qwen3-Embedding-4B",   params_b=4.0, role="scale test (rho)"),
}

MIN_ROWS_PER_DIM = 5.0        # the project's own floor (A1, E1.1) - REGRESSION fits only
USE_BF16 = True               # Qwen3 is trained in bf16; fp16 can overflow silently
NO_INSTRUCTION_PREFIX = True  # captions are document-like; matches bge-m3's use in E1.1

# Existing encoders' parameter counts, for the capacity-ladder plot.
# (approximate, in billions — used only for the x-axis)
KNOWN_PARAMS = {
    "img_small": 0.022, "img_base": 0.087, "img_large": 0.304, "img_giant": 1.14,
    "txt_bge": 0.568, "txt_bert": 0.110, "txt_gpt2": 0.124, "txt_sbert": 0.110,
    "convnext": 0.089, "txt_qwen06": 0.6, "txt_qwen4": 4.0,
}

BATCH_IMG, BATCH_TXT = 32, 64
print("output dir:", OUT)

In [ ]:
# Cell 2 — PRE-REGISTRATION. Runs before any measurement, writes to disk.
#
# The project's standing rule (report E.9): predictions are written down before
# the numbers exist. This cell is that record. Edit the predictions if you
# disagree with them — but edit them NOW, not after seeing results.

PREREG_PATH = os.path.join(OUT, "prereg_scale_extension.json")

PREREG = {
  "written_before_any_measurement": True,
  "hypothesis": (
    "PRH predicts alignment rises with encoder capacity. The project's own capacity "
    "ladder (+15.3 pts of ceiling per decade, R2 0.949, three rungs) predicts the same "
    "for hub transfer on the image side."
  ),
  "predictions": {
    "P1_image_transfer": (
      "DINOv2-giant transfers at or above DINOv2-large's 92.9 percent of native. "
      "CONFIRMED if >= 92.9. A LOWER value falsifies the naive reading of the capacity "
      "ladder and would be the more interesting result."
    ),
    "P2_image_rho": (
      "DINOv2-giant's raw shape agreement with the text encoders is >= DINOv2-large's "
      "best (0.418 with BERT). CONFIRMED if >= 0.418."
    ),
    "P3_text_rho_SCALE": (
      "THE SCALE TEST. Qwen3-Embedding-4B (4.0B, 7.0x bge-m3) shows HIGHER raw shape "
      "agreement with the image encoders than bge-m3's best cross-modal value of 0.320 "
      "(with DINOv2-large). CONFIRMED if >= 0.320. Note rho fits nothing, so this "
      "model's size carries no statistical penalty here even though its hub-transfer "
      "fit would be underpowered."
    ),
    "P3b_text_rho_CONTROL": (
      "ARCHITECTURE CONTROL, not a scale test. Qwen3-Embedding-0.6B is 0.600B against "
      "bge-m3's 0.568B - a 1.06x ratio, i.e. the same scale. Any difference between "
      "them is architecture and training data, NOT capacity. Recorded so that a "
      "difference here is not misread as a scale effect."
    ),
    "P4_decoupling": (
      "Following E.12, rho and transfer need NOT move together. If one rises and the "
      "other does not, that REPLICATES E.12 rather than contradicting anything."
    ),
  },
  "encoder_choice_note": (
    "Qwen3-Embedding-0.6B was initially selected for the scale test on a power "
    "argument, then found to be 1.06x bge-m3 - the same scale, not a rung up. It is "
    "retained as an architecture control and the 4B model added as the actual scale "
    "test. The power objection that excluded 4B applies only to the hub-transfer fit, "
    "not to rho, which fits nothing."
  ),
  "dtype_note": (
    "Text models load in bf16 where the GPU supports it. Qwen3 is trained in bf16 and "
    "fp16 can overflow silently in some activations; a finite-value check runs after "
    "encoding."
  ),
  "prefix_note": (
    "No instruction prefix is applied. Qwen3-Embedding uses instructions on the QUERY "
    "side only; COCO captions are document-like, and bge-m3 is used without a prefix "
    "in E1.1, so omitting it keeps the arms comparable."
  ),
  "pooling_note": (
    "Qwen3-Embedding is a CAUSAL embedding model trained to emit the sentence "
    "representation in the final EOS token, with left padding. It is pooled that "
    "way here. GPT-2 and BERT in the existing hub are masked-mean pooled, which is "
    "correct for them and is what E1.1 does. Pooling is therefore matched to the "
    "model family, not held constant across families - holding it constant would "
    "mean using at least one model wrongly. Set POOL_OVERRIDE in Cell 6 to compare."
  ),
  "threshold_note": (
    "The project treats a 2-point move in transfer as the noise floor (see G10). "
    "Differences below 2 points are not claimed as effects."
  ),
  "what_would_falsify": (
    "P1 falsified if giant transfers below 90.9 percent (more than 2 points under large). "
    "P3 falsified if the larger text encoder's best cross-modal rho is below 0.320."
  ),
}

if os.path.exists(PREREG_PATH):
    PREREG = json.load(open(PREREG_PATH))
    print("Pre-registration already on disk (not overwritten):")
else:
    json.dump(PREREG, open(PREREG_PATH, "w"), indent=2)
    print("Pre-registration WRITTEN to", PREREG_PATH)
for k, v in PREREG["predictions"].items():
    print(f"\n  {k}: {v}")

In [ ]:
# Cell 3 - load the frozen hub, head and encoder caches
#
# THE TRAP THIS AVOIDS. The project has more than one hub:
#   hub_exact.npz   = the PUBLISHED 4-space hub, 512-d, head (512 -> 1024)
#   hub_rebuilt.npz = the 7-space rebuild, whose H_all is 768-d
# Taking W_HEAD from one and H from the other makes Cell 8 compute
# (N,768) @ (512,1024) and die with a gufunc error five cells later.
# load_hub() pairs them from ONE file and asserts the contract here.

def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)

# raw per-encoder vectors - hub-independent, so any source is fine
zr = np.load(os.path.join(FOLDER, "hub_rebuilt.npz"), allow_pickle=True)
names = [str(x) for x in zr["encoder_names"]]
raw   = {n: np.asarray(zr[f"raw_{n}"]) for n in names}
N     = raw[names[0]].shape[0]
print("existing encoders:", names, "| N =", N)

# hub_ids: COCO image id per row IF recorded. Never fall back to arange(N) -
# row indices are not image ids, and comparing them to real ids produces a
# meaningless 'overlap' that reads as an alignment failure.
hub_ids = np.asarray(zr["hub_ids"]).astype(np.int64) if "hub_ids" in zr.files else None
if hub_ids is None:
    print("  note: no hub_ids recorded; order is verified empirically in Cell 7")

def load_hub(path):
    """Return a consistent (H, head, split) set from ONE file, or None."""
    if not os.path.exists(path):
        return None
    z = np.load(path, allow_pickle=True)
    head = None
    for k in ("head", "W_head", "w_head", "caption_head"):
        if k in z.files:
            head = np.asarray(z[k]); break
    if head is None:
        return None
    K = head.shape[0]                      # hub width this head expects

    H = None
    for k in ("H_all", "hub_all", "hub", "H", "hub_train"):
        if k in z.files:
            c = np.asarray(z[k])
            if c.ndim == 2 and c.shape[1] == K:
                H = c
                if c.shape[0] == N:
                    break
    if H is None or H.shape[0] != N:
        # only train rows stored - rebuild all N from the basis, reproducing
        # G0's scalar rescale exactly
        basis = np.asarray(z["basis"]) if "basis" in z.files else None
        mu    = np.asarray(z["mu"])    if "mu"    in z.files else None
        mem = None
        for k in ("space_names", "members", "encoder_names"):
            if k in z.files:
                mem = [str(x) for x in z[k]]; break
        if basis is None or mu is None or mem is None:
            print(f"  {os.path.basename(path)}: head {head.shape} but no {K}-d "
                  f"hub for all {N} rows and cannot rebuild")
            return None
        parts = []
        for m in mem:
            if m not in raw:
                print(f"  cannot rebuild: member {m!r} absent from raw caches")
                return None
            v = np.asarray(raw[m], dtype=np.float64)
            parts.append((v - v.mean(0)) / (v.std(0).mean() + 1e-12))
        Z = np.concatenate(parts, axis=1)
        assert Z.shape[1] == basis.shape[0],             f"concat {Z.shape[1]} != basis rows {basis.shape[0]}"
        H = (Z - mu) @ basis
        print(f"  rebuilt H {H.shape} from basis {basis.shape} over {mem}")

    return dict(H=H, head=head,
                train=np.asarray(z["train_idx"]) if "train_idx" in z.files else None,
                eval=np.asarray(z["eval_idx"])  if "eval_idx"  in z.files else None,
                alpha=float(z["alpha"]) if "alpha" in z.files else 1e-2,
                tag=os.path.basename(path))

HUB = None
for cand in ("hub_exact.npz", "hub_rebuilt.npz"):     # prefer the published hub
    HUB = load_hub(os.path.join(FOLDER, cand))
    if HUB is not None:
        break
assert HUB is not None, "no file supplied a consistent (hub, head) pair"

H_ALL, W_HEAD = HUB["H"], HUB["head"]
TRAIN_IDX, EVAL_IDX, ALPHA = HUB["train"], HUB["eval"], HUB["alpha"]

# assert the contract HERE, not as a matmul error in Cell 8
assert H_ALL.shape[0] == N, f"H has {H_ALL.shape[0]} rows, expected {N}"
assert H_ALL.shape[1] == W_HEAD.shape[0],     f"hub width {H_ALL.shape[1]} != head input {W_HEAD.shape[0]}"
print(f"")
print(f"hub source : {HUB['tag']}")
print(f"  H        : {H_ALL.shape}")
print(f"  W_head   : {W_HEAD.shape}  ({H_ALL.shape[1]} -> {W_HEAD.shape[1]})")
print(f"  train/eval: {len(TRAIN_IDX)} / {len(EVAL_IDX)} | alpha {ALPHA}")

T_BGE = np.asarray(zr["raw_txt_bge"])
print(f"  T_BGE    : {T_BGE.shape}")

CONV = os.path.join(FOLDER, "e1_img_ckpt_convnext-base-224-22k_native.npz")
if os.path.exists(CONV):
    zc = np.load(CONV, allow_pickle=True)
    for k in zc.files:
        a = np.asarray(zc[k])
        if a.ndim == 2 and a.shape[0] == N and a.shape[1] >= 512:
            raw["convnext"] = a
            if "convnext" not in names: names.append("convnext")
            print(f"  ConvNeXt : {a.shape}")
            break

print("")
print(f"roster after this cell: {len(names)} encoders - {names}")
print("NOTE: this cell REBUILDS the roster. Cells 5 and 6 add DINOv2-giant and")
print("      the Qwen models back. Do not skip them, even when their caches")
print("      already exist - Cell 7 will refuse to run on a partial roster.")

LAB = {"img_small":"DINOv2-small","img_base":"DINOv2-base","img_large":"DINOv2-large",
       "img_giant":"DINOv2-giant","txt_bge":"bge-m3","txt_gpt2":"GPT-2","txt_bert":"BERT",
       "txt_sbert":"SBERT","convnext":"ConvNeXt-base",
       "txt_qwen06":"Qwen3-Emb-0.6B","txt_qwen4":"Qwen3-Emb-4B"}
def lab(n): return LAB.get(n, n)
IMG_DINO = {"img_small","img_base","img_large","img_giant"}
IMG_ALL  = IMG_DINO | {"convnext"}
TXT_ALL  = set(names) - IMG_ALL

In [ ]:
# Cell 4 - resolve the same COCO items, using E1.1's own procedure
#
# Taken from E1_1_text_encoder_pair.ipynb so the new encoders see EXACTLY the
# items the hub was built from:
#   * annotations_trainval2017.zip, member annotations/captions_train2017.json
#   * ids = sorted(set(captions) & set(urls))[:N_PAIRS]   - sorted, not random
#   * images fetched individually over coco_url with a thread pool
#   * ALL captions per image (averaged later), MAX_LEN 64
#
# N_PAIRS in E1.1 is derived from encoder width:
#     N_PAIRS = N_EVAL + 10 * width / 0.9
# which reproduces the cache sizes exactly: small cls+patch 768d -> 9,533;
# base 1536d -> 18,066; large 2048d -> 23,755. The HUB uses 9,533, set by the
# narrowest source. This notebook therefore takes the first 9,533 sorted ids -
# the same set, in the same order, as hub_rebuilt.npz.

import json as _json, zipfile, urllib.request, io, time
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from pathlib import Path

ANN_URL = ("http://images.cocodataset.org/annotations/"
           "annotations_trainval2017.zip")
ANN     = Path(FOLDER) / "annotations_trainval2017.zip"
MEMBER  = "annotations/captions_train2017.json"
MAX_LEN = 64
ALL_CAPTIONS = True

def _usable(path):
    if not path.exists() or path.stat().st_size < 100_000_000:
        return False
    try:
        with zipfile.ZipFile(path) as z:
            return MEMBER in z.namelist()
    except zipfile.BadZipFile:
        return False

if _usable(ANN):
    print(f"using cached annotations ({ANN.stat().st_size/1e6:.0f} MB)")
else:
    if ANN.exists():
        print("cached file truncated or corrupt - re-downloading")
        ANN.unlink()
    print(f"downloading annotations (~250 MB, once) ...")
    tmp = ANN.with_suffix(".part")
    urllib.request.urlretrieve(ANN_URL, str(tmp))
    tmp.rename(ANN)
    assert _usable(ANN), "download completed but the archive is unreadable"

with zipfile.ZipFile(ANN) as z:
    with z.open(MEMBER) as f:
        ann = _json.load(f)

caps = {}
for a in ann["annotations"]:
    caps.setdefault(a["image_id"], []).append(a["caption"].strip())
cap_sel = {k: (v if ALL_CAPTIONS else v[:1]) for k, v in caps.items()}
url = {im["id"]: im["coco_url"] for im in ann["images"]}

ids_all  = sorted(set(cap_sel) & set(url))
IDS      = ids_all[:N]                 # N = 9,533, the hub corpus
CAPTIONS = [cap_sel[i] for i in IDS]
print(f"{len(IDS)} image/caption groups (from {len(ids_all)} available)")

# --- alignment check against the hub's own ids, if recorded ---
# --- NO id comparison here, and this is deliberate ---
# hub_rebuilt.npz's `hub_ids` are ROW INDICES (0..N-1), not COCO image ids:
# G0_hub_rebuild derives them from the DINOv2 caches' `keep` arrays, which
# E1.1 records as request positions. ConvNeXt/SigLIP caches store real COCO
# ids instead. The two namespaces cannot be joined, and an "overlap" between
# them (1,879) is just how many COCO ids fall below 9,533 - G4_convnext
# documents exactly this. An earlier version of this cell reproduced that
# noise and reported it as a misalignment.
#
# Alignment holds because every cache was written by the SAME deterministic
# loop over sorted(set(caps) & set(url))[:N]. It is verified EMPIRICALLY in
# Cell 7: a new encoder must correlate far more with an existing one than
# with a shuffled copy of it (H1 check #2). That is the only test that
# applies across these caches.
print("  alignment: positional, by shared deterministic ordering;")
print("  verified empirically in Cell 7 (shuffle check), not by id join.")

CAP_COUNTS = [len(c) for c in CAPTIONS]
print(f"captions per item: min {min(CAP_COUNTS)}, max {max(CAP_COUNTS)}, "
      f"mean {sum(CAP_COUNTS)/len(CAP_COUNTS):.2f}")

# --- image fetcher, as in E1.1 (individual coco_url GETs, threaded) ---
WORKERS, CHUNK, BATCH_FETCH = 32, 512, 64

def fetch_one(n):
    try:
        with urllib.request.urlopen(url[IDS[n]], timeout=8) as r:
            return n, Image.open(io.BytesIO(r.read())).convert("RGB")
    except Exception:
        return n, None

print("")
print("Image bytes are fetched per-item from coco_url at encode time,")
print("exactly as E1.1 does - no train2017.zip is needed (that is 19 GB).")
print("Cell 5 uses fetch_one() and records a `keep` array for any that fail.")

In [ ]:
# Cell 5 - encode the new IMAGE model(s), following E1.1 exactly:
# fp16 on GPU, cls or cls+patch pooling, threaded coco_url fetch, resumable
# checkpoint that records its own tag so a pooling change cannot silently
# corrupt a resumed cache.
from transformers import AutoImageProcessor, AutoModel

def encode_image_model(key, cfg):
    pooling = cfg.get("pooling", "cls+patch")
    tag  = cfg["hf"].split("/")[-1] + "_" + pooling
    ckpt = os.path.join(OUT, f"cache_{key}.npz")

    if os.path.exists(ckpt):
        d = np.load(ckpt, allow_pickle=True)
        if str(d.get("tag", "")) == tag and int(d.get("next", 0)) >= N:
            print(f"[{key}] cached: {d['v'].shape} ({tag})")
            return np.asarray(d["v"]), np.asarray(d["keep"])
        print(f"[{key}] checkpoint tag mismatch or incomplete - restarting")

    proc = AutoImageProcessor.from_pretrained(cfg["hf"])
    vis  = AutoModel.from_pretrained(cfg["hf"]).to(DEV).eval()
    # fp16 DELIBERATELY, not bf16 - unlike the text cell.
    #   * E1.1 encodes DINOv2 small/base/large with .half(); the existing
    #     three ladder rungs are all fp16 caches.
    #   * Putting giant on bf16 would place one rung on a different numeric
    #     path from the other three, on the very axis being measured.
    #   * The bf16 switch in Cell 6 is Qwen-specific: Qwen3 is TRAINED in
    #     bf16 and its activations can exceed fp16 range. DINOv2 is not.
    # Giant is 3.7x larger than any DINOv2 run here before, so the guard
    # below checks rather than assumes.
    if DEV == "cuda":
        vis = vis.half()

    img_list, keep, t0 = [], [], time.time()
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        for c0 in range(0, N, CHUNK):
            c1 = min(c0 + CHUNK, N)
            got = sorted((r for r in pool.map(fetch_one, range(c0, c1))
                          if r[1] is not None), key=lambda x: x[0])
            for b0 in range(0, len(got), BATCH_FETCH):
                batch = got[b0:b0 + BATCH_FETCH]
                with torch.no_grad():
                    x = proc(images=[im for _, im in batch], return_tensors="pt").to(DEV)
                    if DEV == "cuda":
                        x["pixel_values"] = x["pixel_values"].half()
                    o = vis(**x).last_hidden_state
                    h = (o[:, 0] if pooling == "cls"
                         else torch.cat([o[:, 0], o[:, 1:].mean(1)], dim=-1))
                img_list.append(h.float().cpu().numpy())
                keep += [n for n, _ in batch]
            rate = (c1) / max(time.time() - t0, 1e-9)
            print(f"  {c1}/{N} kept {len(keep)} {rate:.0f} img/s "
                  f"ETA {(N-c1)/max(rate,1e-9)/60:.1f} min", flush=True)
            np.savez_compressed(ckpt,
                                v=np.concatenate(img_list).astype(np.float32),
                                keep=np.array(keep), next=c1,
                                tag=np.array(tag), params_b=cfg["params_b"])
    del vis
    torch.cuda.empty_cache()
    V = np.concatenate(img_list).astype(np.float64)

    # dtype guard - an fp16 overflow shows up here, never as an exception.
    n_bad = int((~np.isfinite(V)).sum())
    if n_bad:
        raise RuntimeError(
            f"{n_bad} non-finite values in {key} ({cfg['hf']}) at fp16. "
            "Re-run this model with torch.float32 and RECORD the dtype change - "
            "it puts this rung on a different numeric path from the fp16 "
            "small/base/large caches, which must be disclosed when the ladder "
            "is reported.")
    norms = np.linalg.norm(V, axis=1)
    print(f"[{key}] done: {V.shape} ({pooling}), kept {len(keep)}/{N}")
    print(f"    finite OK | row norm min {norms.min():.2f} "
          f"max {norms.max():.2f} mean {norms.mean():.2f}")
    if norms.max() > 1e4 or norms.min() < 1e-4:
        print("    WARNING: extreme row norms - inspect before trusting this cache.")
    return V, np.array(keep)

KEEP = {}
for key, cfg in IMAGE_MODELS.items():
    V, kp = encode_image_model(key, cfg)
    raw[key] = V
    KEEP[key] = kp
    if key not in names:
        names.append(key)
    exp = cfg["native_dim"] * (2 if cfg.get("pooling","cls+patch")=="cls+patch" else 1)
    print(f"    width {V.shape[1]} (expected {exp}), "
          f"rows/dim {len(TRAIN_IDX)/V.shape[1]:.1f}")

In [ ]:
# Cell 6 - encode the new TEXT model(s).
#
# POOLING IS NOT ONE-SIZE-FITS-ALL, and getting it wrong fails silently.
#
#   E1.1 masked-mean-pools GPT-2 and BERT. That is correct for those:
#   BERT is bidirectional, and GPT-2 has no trained sentence token, so the
#   mean over positions is the standard choice.
#
#   Qwen3-Embedding is different. It is a CAUSAL model fine-tuned for
#   embeddings, trained to place the sentence representation in the final
#   EOS token, with LEFT padding so that token is last in every row.
#   Mean-pooling it returns a valid tensor and raises nothing - it is just
#   not the representation the model was trained to emit. Using it would
#   confound 'does scale help' with 'did I pool it wrong'.
#
# So pooling is selected per model. Set POOL_OVERRIDE to compare.

from transformers import AutoTokenizer, AutoModel as HFAutoModel

POOL_OVERRIDE = None        # None = auto; or "mean" / "lasttoken" to force

def pooling_for(hf_name):
    if POOL_OVERRIDE:
        return POOL_OVERRIDE
    n = hf_name.lower()
    if "qwen" in n or "e5-mistral" in n or "gte-qwen" in n:
        return "lasttoken"       # causal embedding models
    return "mean"                 # BERT/GPT-2-style, as E1.1 does

def encode_text_model(key, cfg):
    ckpt = os.path.join(OUT, f"cache_{key}.npz")
    pool = pooling_for(cfg["hf"])
    if os.path.exists(ckpt):
        dd = np.load(ckpt, allow_pickle=True)
        if str(dd.get("pool", "")) == pool:
            print(f"[{key}] cached: {dd['v'].shape} (pool={pool})")
            return np.asarray(dd["v"])
        print(f"[{key}] cached with a different pooling - re-encoding")

    tok = AutoTokenizer.from_pretrained(cfg["hf"])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    if pool == "lasttoken":
        tok.padding_side = "left"      # puts EOS last in every row
    # bf16 where supported: Qwen3 is trained in bf16 and fp16 can overflow in
    # some activations, producing inf/nan with no error raised.
    dtype = torch.bfloat16 if (USE_BF16 and DEV == "cuda"
                               and torch.cuda.is_bf16_supported()) else torch.float16
    lm = HFAutoModel.from_pretrained(cfg["hf"], torch_dtype=dtype).to(DEV).eval()
    print(f"[{key}] {cfg['hf']} | pooling = {pool} | padding = {tok.padding_side} "
          f"| dtype = {str(dtype).split('.')[-1]} | role = {cfg.get('role','-')}")

    flat, owner = [], []
    for j, group in enumerate(CAPTIONS):
        for c in group:
            flat.append(c); owner.append(j)
    owner = np.asarray(owner)
    print(f"    {len(flat)} captions for {N} items")

    out = []
    for b in range(0, len(flat), BATCH_TXT):
        enc = tok(flat[b:b+BATCH_TXT], return_tensors="pt", padding=True,
                  truncation=True, max_length=MAX_LEN).to(DEV)
        with torch.no_grad():
            h = lm(**enc).last_hidden_state
        if pool == "lasttoken":
            pooled = h[:, -1]                       # left-padded, so EOS is last
        else:
            m = enc["attention_mask"].unsqueeze(-1).to(h.dtype)
            pooled = (h * m).sum(1) / m.sum(1).clamp(min=1)
        out.append(pooled.float().cpu().numpy())
        if (b // BATCH_TXT) % 50 == 0:
            print(f"    {b}/{len(flat)}", flush=True)
    del lm; torch.cuda.empty_cache()

    E = np.concatenate(out).astype(np.float64)
    # dtype guard - an fp16 overflow shows up here, not as an exception
    n_bad = int((~np.isfinite(E)).sum())
    if n_bad:
        raise RuntimeError(f"{n_bad} non-finite values in {key} embeddings - "
                           f"likely an fp16 overflow. Set USE_BF16=True or force float32.")
    norms = np.linalg.norm(E, axis=1)
    print(f"    finite OK | norm min {norms.min():.2f} max {norms.max():.2f} "
          f"mean {norms.mean():.2f}")
    V = np.zeros((N, E.shape[1]), dtype=np.float64)
    for j in range(N):
        V[j] = E[owner == j].mean(0)                # average the item's captions
    np.savez_compressed(ckpt, v=V, hf=cfg["hf"], params_b=cfg["params_b"], pool=np.array(pool))
    print(f"[{key}] done: {V.shape}")
    return V

for key, cfg in TEXT_MODELS.items():
    V = encode_text_model(key, cfg)
    raw[key] = V
    if key not in names:
        names.append(key)
    print(f"    rows/dim {len(TRAIN_IDX)/V.shape[1]:.1f}")

print("")
print("all encoders now available:", names)

In [ ]:
# Cell 6b - POOLING DIAGNOSTIC. Do not take the pooling choice on trust.
#
# Qwen3-Embedding is causal: only the final token has attended to the whole
# sentence. Mean-pooling averages one trained sentence vector with N-1
# positions that still carry next-token-prediction features. The failure is
# SILENT - a valid tensor, no warning, just a worse representation.
#
# This project already measured what that looks like. GPT-2, mean-pooled
# (C.11): effective rank 6.6 of 768, mean pairwise cosine +0.999, readable
# at 95.1 percent but writable at only 24.8. A cone, not a cloud.
#
# So: encode a sample BOTH ways and run the same two diagnostics. If the
# mean-pooled version is collapsed and the last-token version is not, the
# pooling choice is settled by measurement rather than by assertion.

def spectrum_stats(V, name=""):
    """effective rank (entropy of the normalised spectrum) + mean pairwise cosine"""
    X = l2n(np.asarray(V, dtype=np.float64))
    # UNCENTRED spectrum, as C.11 does. Centring would subtract the dominant
    # shared direction - which is precisely the thing being diagnosed - and a
    # collapsed space would then score the same effective rank as a healthy one.
    s = np.linalg.svd(X, compute_uv=False)
    p = (s**2) / (s**2).sum()
    p = p[p > 0]
    eff_rank = float(np.exp(-(p * np.log(p)).sum()))
    m = min(len(X), 1500)
    S = X[:m] @ X[:m].T
    iu = np.triu_indices(m, 1)
    cos = float(S[iu].mean())
    print(f"    {name:28s} dim {X.shape[1]:5d} | eff.rank {eff_rank:7.1f} "
          f"| mean pairwise cos {cos:+.3f}")
    return eff_rank, cos

print("=" * 72)
print("POOLING DIAGNOSTIC  (C.11's two measures)")
print("=" * 72)
print("")
print("  reference points already in the project:")
print("    GPT-2 mean-pooled (collapsed)   eff.rank    6.6 | mean cos +0.999")
print("    BERT                            eff.rank   65.0 | healthy")
print("    SBERT / bge-m3                  eff.rank   93+  | healthy")
print("")

SAMPLE = 1500       # enough to see collapse; keeps this cell quick
for key, cfg in TEXT_MODELS.items():
    print(f"  {lab(key)} - {cfg['hf']}")
    saved = POOL_OVERRIDE
    results = {}
    for pool in ("mean", "lasttoken"):
        globals()["POOL_OVERRIDE"] = pool
        tok = AutoTokenizer.from_pretrained(cfg["hf"])
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        if pool == "lasttoken":
            tok.padding_side = "left"
        lm = HFAutoModel.from_pretrained(cfg["hf"], torch_dtype=torch.float16).to(DEV).eval()
        firsts = [g[0] for g in CAPTIONS[:SAMPLE]]      # one caption per item
        out = []
        for b in range(0, len(firsts), BATCH_TXT):
            enc = tok(firsts[b:b+BATCH_TXT], return_tensors="pt", padding=True,
                      truncation=True, max_length=MAX_LEN).to(DEV)
            with torch.no_grad():
                h = lm(**enc).last_hidden_state
            if pool == "lasttoken":
                pooled = h[:, -1]
            else:
                mk = enc["attention_mask"].unsqueeze(-1).to(h.dtype)
                pooled = (h * mk).sum(1) / mk.sum(1).clamp(min=1)
            out.append(pooled.float().cpu().numpy())
        del lm; torch.cuda.empty_cache()
        results[pool] = spectrum_stats(np.concatenate(out), f"pooled by {pool}")
    globals()["POOL_OVERRIDE"] = saved

    er_mean, cos_mean = results["mean"]
    er_last, cos_last = results["lasttoken"]
    print("")
    if er_last > er_mean * 2.0 or cos_last < cos_mean - 0.3:
        print(f"    -> last-token pooling gives the healthier geometry "
              f"(eff.rank {er_last:.0f} vs {er_mean:.0f}). Using it, as configured.")
    elif er_mean > er_last * 2.0:
        print(f"    -> UNEXPECTED: mean pooling looks healthier here. Investigate")
        print(f"       before trusting this encoder's rho - set POOL_OVERRIDE='mean'.")
    else:
        print(f"    -> the two poolings are comparable on these measures; the")
        print(f"       choice is unlikely to drive the result either way.")
    print("")

In [ ]:
# Cell 7 - MEASUREMENT 1: the shape-agreement matrix, EXTENDED
#
# This is the same Spearman-of-pairwise-distances test as the main figures
# notebook, now over 11 encoders instead of 8:
#
#     7 hub members  +  ConvNeXt-base (held out)
#   + DINOv2-giant        (image side, cls+patch to match the ladder)
#   + Qwen3-Embedding-0.6B (text side, architecture control)
#   + Qwen3-Embedding-4B   (text side, the scale test)
#   = 11 encoders, C(11,2) = 55 pairs
#   (the count is computed from `names` below, so it stays correct if the
#    roster changes again - this comment is the thing that goes stale)
#
# No map is fitted anywhere. This measures raw geometry only, so it is the
# PRH-style alignment measure and is independent of the hub entirely.
#
# DTYPES - three different ones, doing three different jobs. Do not confuse
# them with the fp16/bf16 used when ENCODING:
#   fp16 / bf16 : the ENCODER forward pass (Cells 5-6), on GPU, for speed.
#                 Vectors are cast up and stored as float32.
#   float64     : this cell's ranking step. C(9533,2) = 45,434,278 pairwise
#                 distances, so the ranks run to 45.4M. float32's 24-bit
#                 mantissa is exact only to 16,777,216 - above that it skips
#                 integers, and two different ranks can collide on one float,
#                 silently corrupting every correlation with no error raised.
#   float32     : safe only AFTER centring and unit-normalising, which pulls
#                 every value into [-1, 1] where float32 has ample precision.
# Order matters: rank in float64, centre, normalise, THEN cast. Each Spearman
# is then a single dot product, exact to ~1e-9 against scipy.stats.spearmanr.

from scipy.spatial.distance import pdist
from scipy.stats import rankdata

def rank_vec(X):
    d = pdist(l2n(X), metric="cosine")
    r = rankdata(d).astype(np.float64); del d
    r -= r.mean(); r /= np.linalg.norm(r)
    return r.astype(np.float32)

# --- ROSTER GUARD -----------------------------------------------------
# Cell 3 REBUILDS `raw` and `names` from the hub caches alone. Cells 5 and 6
# are what append the new encoders. Running 3 and then jumping here silently
# drops them, and every number below is then computed over a smaller roster
# that still looks perfectly plausible - 28 pairs instead of 55, an empty
# "NEW ENCODERS" section, and a verdict cell with nothing to score.
EXPECTED = list(IMAGE_MODELS) + list(TEXT_MODELS)
missing  = [k for k in EXPECTED if k not in names]
if missing:
    raise RuntimeError(
        "ROSTER INCOMPLETE - missing: " + ", ".join(missing) + ". "
        "Cell 3 resets the roster; Cells 5 and 6 add these back and load "
        "from cache in seconds, no encoding. Run 5 and 6, then re-run this "
        "cell. Refusing to compute a matrix that would look valid but omit "
        "the encoders this notebook exists to test.")
print(f"roster OK: {len(names)} encoders, {len(names)*(len(names)-1)//2} pairs")

ranks = {}
for n_ in names:
    ranks[n_] = rank_vec(raw[n_])
    print("    ranked", lab(n_))

K = len(names); RHO = np.eye(K)
for i in range(K):
    for j in range(i+1, K):
        RHO[i, j] = RHO[j, i] = float(np.dot(ranks[names[i]], ranks[names[j]]))
del ranks
np.savez(os.path.join(OUT, "rho_matrix_extended.npz"), names=names, M=RHO, n_items=N)

# --- ALIGNMENT CHECK (H1 check #2, adapted to rho) ---------------------
# A newly encoded encoder must agree far more with an existing one than
# with a row-shuffled copy of it. If the new caches were built on a
# different item order, every rho above is meaningless - and would look
# plausible rather than raising anything.
NEWKEYS_ = [k for k in list(IMAGE_MODELS) + list(TEXT_MODELS) if k in names]
if NEWKEYS_:
    ref = "txt_bge" if "txt_bge" in names else names[0]
    print("")
    print("alignment check - new encoders vs an existing cache:")
    for k in NEWKEYS_:
        rt = RHO[names.index(k), names.index(ref)]
        Xs = raw[k][np.random.default_rng(0).permutation(N)]
        rs = float(np.dot(rank_vec(Xs), rank_vec(raw[ref])))
        ok = (rt - rs) > 0.05
        print(f"  {lab(k):16s} vs {lab(ref):10s} aligned {rt:+.3f} | "
              f"shuffled {rs:+.3f} | gap {rt-rs:+.3f}  "
              f"{'OK' if ok else 'MISALIGNED - do not trust these rho values'}")

IMG = {"img_small","img_base","img_large","img_giant","img_giant_cp","convnext"}
def best_cross(n_):
    i = names.index(n_); same = n_ in IMG
    c = [(RHO[i,j], names[j]) for j in range(K) if j != i and ((names[j] in IMG) != same)]
    return max(c) if c else (float("nan"), None)

# ---- heatmap ----
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7.0, 6.2), dpi=200)
im = ax.imshow(RHO, cmap="RdYlBu_r", vmin=0, vmax=1)
labels = [lab(n) for n in names]
ax.set_xticks(range(K)); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
ax.set_yticks(range(K)); ax.set_yticklabels(labels, fontsize=7)
for i in range(K):
    for j in range(K):
        ax.text(j, i, f"{RHO[i,j]:.2f}", ha="center", va="center", fontsize=5.6,
                color="white" if (RHO[i,j] > 0.6 or RHO[i,j] < 0.2) else "black")
NEWKEYS = [k for k in list(IMAGE_MODELS) + list(TEXT_MODELS) if k in names]
ax.set_title("Shape agreement - Spearman of pairwise distances" + chr(10) +
             "%d pairs, no map fitted, all %d items (+ %s)"
             % (K*(K-1)//2, N, ", ".join(lab(k) for k in NEWKEYS)))
fig.colorbar(im, ax=ax, fraction=0.046).set_label("rank correlation rho")
fig.tight_layout()
fig.savefig(os.path.join(OUT, "rho_matrix_10encoders.png"))
plt.show(); plt.close(fig)

# ---- what the new encoders did ----
print("")
print("=" * 66)
print("NEW ENCODERS - raw shape agreement")
print("=" * 66)
for key in NEWKEYS:
    i = names.index(key)
    v, partner = best_cross(key)
    print("")
    print(f"  {lab(key)}  ({raw[key].shape[1]}d)")
    print(f"    best cross-modal rho = {v:.3f}  with {lab(partner)}")
    for j in range(K):
        if j != i:
            print(f"      vs {lab(names[j]):18s} {RHO[i,j]:.3f}")

print("")
print("=" * 66)
print("REFERENCE - the values these are being compared against")
print("=" * 66)
for n_ in ["img_large","txt_bge","txt_sbert","convnext"]:
    if n_ in names:
        v, p_ = best_cross(n_)
        print(f"  {lab(n_):18s} best cross-modal rho = {v:.3f}  with {lab(p_)}")
cm = [RHO[i,j] for i in range(K) for j in range(i+1,K)
      if (names[i] in IMG) != (names[j] in IMG)]
print("")
print(f"  cross-modal mean over all {len(cm)} image-text pairs = {sum(cm)/len(cm):.3f}")
print("  (the 7-encoder + ConvNeXt baseline was 0.290)")

In [ ]:
# Cell 8 — MEASUREMENT 2: hub transfer through the FROZEN hub and FROZEN head
#
# Exactly the ConvNeXt protocol (report E.2 / the frozen-hub document):
#   1. fit ONE entry map  X_new -> H   on the 8,533 training rows  (closed form)
#   2. apply the EXISTING frozen head  H -> bge   unchanged        (zero-shot)
#   3. score R@1 on the 1,000 held-out rows against a natively-fitted head
# The hub is not rebuilt and the head is not refitted. Nothing upstream moves.

def ridge(A, B, alpha=ALPHA):
    return np.linalg.solve(A.T @ A + alpha*np.eye(A.shape[1]), A.T @ B)

def r_at_1(P, G, return_hits=False):
    sims = l2n(P) @ l2n(G).T
    hits = (np.argmax(sims, 1) == np.arange(len(P)))
    return (float(hits.mean()), hits) if return_hits else float(hits.mean())

def noise_test(hits_a, hits_b, label_a="transfer", label_b="native",
               B=5000, seed=0):
    """Is the gap between two R@1 figures distinguishable from noise?

    R@1 is a per-item BINARY outcome scored on the SAME 1,000 items, so the
    comparison is PAIRED and an unpaired test would be wrong. Two criteria,
    both fixed before looking:

      1. McNemar exact test on the discordant items - those one system got
         right and the other got wrong. Under the null those two counts are
         equal. p >= 0.05 means the split is consistent with chance.
      2. Bootstrap 95% CI on the ratio, resampling ITEMS. If the interval
         contains 100 percent, the two systems are not separated.

    NOISE requires BOTH. Either one alone can mislead: a tiny consistent
    effect can clear McNemar on large n, and a wide CI can hide a real one.
    """
    from scipy.stats import binomtest
    a_only = int((hits_a & ~hits_b).sum())
    b_only = int((~hits_a & hits_b).sum())
    p = 1.0 if (a_only + b_only) == 0 else         binomtest(a_only, a_only + b_only, 0.5).pvalue
    r = np.random.default_rng(seed); n = len(hits_a); ratios = np.empty(B)
    for i in range(B):
        s = r.integers(0, n, n)
        ratios[i] = 100.0 * hits_a[s].mean() / max(hits_b[s].mean(), 1e-9)
    lo, hi = np.percentile(ratios, [2.5, 97.5])
    is_noise = (p >= 0.05) and (lo <= 100.0 <= hi)
    return dict(a_only=a_only, b_only=b_only, p=float(p),
                ci_lo=float(lo), ci_hi=float(hi), is_noise=bool(is_noise))

GAL = T_BGE[EVAL_IDX]           # held-out bge gallery
transfer = {}

# CONTROL CHOICE. img_large would be the wrong control: it is a MEMBER of
# the 4-space hub, so refitting its entry map partly recovers it from a
# basis built out of itself - it reads ~99% where the published zero-shot
# figure is 92.9%, and the two are not the same quantity. ConvNeXt and
# SigLIP were never hub members, so their entry maps are genuine out-of-hub
# fits and their published values (96.7 / 94.2) are directly comparable.
CONTROLS = [c for c in ("convnext", "siglip2") if c in raw]
if not CONTROLS:
    print("NOTE: no out-of-hub control available (ConvNeXt/SigLIP not loaded).")
    print("      Transfer numbers below have no calibrated reference.")
IN_HUB = set(str(x) for x in zr["encoder_names"]) if "encoder_names" in zr.files else set()

for key in list(IMAGE_MODELS) + list(TEXT_MODELS) + CONTROLS:
    if key not in raw: continue
    X = np.asarray(raw[key], dtype=np.float64)

    # 1. entry map into the frozen hub, fitted on TRAIN rows only
    W_entry = ridge(X[TRAIN_IDX], H_ALL[TRAIN_IDX])
    H_new   = X @ W_entry

    # 2. frozen head, unchanged -> predictions on held-out rows
    pred_transfer = H_new[EVAL_IDX] @ W_HEAD

    # 3. native reference: a head fitted for THIS encoder specifically
    W_native = ridge(H_new[TRAIN_IDX], T_BGE[TRAIN_IDX])
    pred_native = H_new[EVAL_IDX] @ W_native

    # 4. random-map control
    rng = np.random.default_rng(0)
    W_rand = rng.standard_normal(W_entry.shape) * (W_entry.std())
    pred_rand = (X @ W_rand)[EVAL_IDX] @ W_HEAD

    r_t, hits_t = r_at_1(pred_transfer, GAL, return_hits=True)
    r_n, hits_n = r_at_1(pred_native,  GAL, return_hits=True)
    r_c = r_at_1(pred_rand, GAL)
    pct = 100.0 * r_t / max(r_n, 1e-9)

    # POWER GATE. Ridge below ~5 rows per input dimension does not become
    # noisy - it becomes SYSTEMATICALLY PESSIMISTIC with no error raised.
    # That is the sample-starvation bias recorded in A1's review notes as the
    # cause of Experiment A's failed run 1. A low number from an underpowered
    # fit is not evidence about capacity; it is evidence about the fit.
    rpd = len(TRAIN_IDX) / X.shape[1]
    powered = rpd >= MIN_ROWS_PER_DIM
    transfer[key] = dict(r1_transfer=r_t, r1_native=r_n, r1_control=r_c,
                         pct_of_native=pct, rows_per_dim=rpd, powered=bool(powered),
                         width=int(X.shape[1]))
    # The head predicts bge-m3 caption vectors. For an IMAGE encoder that is
    # a CROSS-MODAL problem; for a TEXT encoder it is text-to-text, which is
    # far easier. The two are NOT comparable and are labelled as such.
    task = "img->txt" if key in IMG_ALL else "txt->txt"
    inhub = " [IN-HUB]" if key in IN_HUB else ""
    transfer[key]["task"] = task
    transfer[key]["in_hub"] = bool(key in IN_HUB)
    flag = "" if powered else "   <-- UNDERPOWERED, INCONCLUSIVE"
    print(f"{lab(key):18s} {task:9s} R@1 transfer {r_t:.3f} | native {r_n:.3f} | "
          f"{pct:5.1f}% of native | control {r_c:.4f} | {rpd:4.1f} rows/dim"
          f"{inhub}{flag}")
    # A frozen head should not BEAT a head fitted for this encoder. When it
    # appears to, test it rather than wave it away.
    if r_t >= r_n:
        nt = noise_test(hits_t, hits_n)
        transfer[key]["noise_test"] = nt
        verdict_txt = ("consistent with NOISE" if nt["is_noise"]
                       else "NOT explained by noise - investigate")
        print(f"{'':18s}   transfer >= native ({pct:.1f}%). Paired test: "
              f"McNemar p={nt['p']:.3f}")
        print(f"{'':18s}     discordant items: transfer-only {nt['a_only']}, "
              f"native-only {nt['b_only']}")
        print(f"{'':18s}     bootstrap 95% CI on ratio: "
              f"[{nt['ci_lo']:.1f}%, {nt['ci_hi']:.1f}%]  "
              f"{'contains' if nt['ci_lo'] <= 100 <= nt['ci_hi'] else 'EXCLUDES'} 100%")
        print(f"{'':18s}     -> {verdict_txt}")
        if not nt["is_noise"] and powered:
            print(f"{'':18s}        A powered fit where the frozen head genuinely")
            print(f"{'':18s}        beats the native one would be a real finding.")
    if r_n > 0 and abs(r_t - r_n) < 1e-3 and not powered:
        print(f"{'':18s}   ^ transfer EQUALS native. At {rpd:.1f} rows/dim both fits")
        print(f"{'':18s}     collapse, so the ratio goes to 1.0 because the")
        print(f"{'':18s}     DENOMINATOR fell - not because transfer is perfect.")
    if not powered:
        print(f"{'':18s}   {X.shape[1]}d needs >= {int(MIN_ROWS_PER_DIM*X.shape[1])} "
              f"train rows for a powered fit; only {len(TRAIN_IDX)} available.")
        print(f"{'':18s}   Do NOT read this as a capacity result - report it as inconclusive.")

def _ser(v):
    """float() numerics, pass str/bool through, RECURSE into dicts.
    The noise_test entry is a nested dict; a one-level serializer would
    stringify it into the JSON and make it unreadable by anything but eye."""
    if isinstance(v, (bool, str)):
        return v
    if isinstance(v, dict):
        return {k: _ser(x) for k, x in v.items()}
    try:
        return float(v)
    except (TypeError, ValueError):
        return str(v)

json.dump({k: {kk: _ser(vv) for kk, vv in v.items()} for k, v in transfer.items()},
          open(os.path.join(OUT, "transfer_extended.json"), "w"), indent=2)

# ---- CALIBRATION, read off the out-of-hub control -------------------
# This cell's protocol is NOT the published one: it refits an entry map into
# a rebuilt H, whereas the published figures come from each encoder's own
# G-series notebook. The control tells us how far apart the two protocols
# sit, so the other rows can be read on the published scale instead of
# being compared to it naively.
PUBLISHED = {"convnext": 96.7, "siglip2": 94.2}
offsets = [(k, transfer[k]["pct_of_native"], PUBLISHED[k])
           for k in CONTROLS if k in transfer and k in PUBLISHED]
if offsets:
    print("")
    print("CALIBRATION vs the published protocol:")
    for k, got, pub in offsets:
        print(f"  {lab(k):16s} here {got:5.1f}%  published {pub:5.1f}%  "
              f"offset {got-pub:+5.1f} pts")
    mean_off = sum(g - p for _, g, p in offsets) / len(offsets)
    print(f"  -> this cell reads {mean_off:+.1f} pts vs the published protocol.")
    print(f"     Subtract that before comparing any row above to 94.2 / 96.7,")
    print(f"     or compare rows only to EACH OTHER within this table.")
    if abs(mean_off) > 3:
        print(f"     The offset is large. Treat absolute values here as")
        print(f"     uncalibrated; the RELATIVE ordering is what this cell")
        print(f"     supports.")
print("")
print("Published zero-shot references (img->txt, out-of-hub): "
      "SigLIP 94.2, ConvNeXt 96.7; within-family band 93.8-96.5 (chance 0.001).")
print("Compare ONLY against rows with the same task label, and treat any")
print("row marked IN-HUB or UNDERPOWERED as uncalibrated.")

In [ ]:
# Cell 9 - the capacity ladder
#
# ONLY rho is plotted against capacity. The transfer panel was removed, for
# three reasons documented in Cell 8:
#   1. the head predicts bge vectors, so image encoders face a CROSS-MODAL
#      problem (R@1 ~0.48) and text encoders a text-to-text one (~0.90);
#      one axis cannot carry both
#   2. any in-hub encoder reads inflated, because its entry map partly
#      recovers it from a basis built out of itself
#   3. wide encoders are underpowered at 8,533 train rows, and a starved
#      fit drives transfer/native toward 1.0 by collapsing the denominator
# rho has none of these problems: it fits nothing, so width and hub
# membership are irrelevant to it. Transfer numbers stay in the printed
# table and the JSON, where their labels travel with them.
import matplotlib.pyplot as plt

NAVY, BLUE, TEAL, RED, GREY = "#172B54", "#2B5FD9", "#0F766E", "#B02418", "#6B7280"
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":8.5,"text.color":NAVY,
                     "axes.labelcolor":NAVY,"xtick.color":NAVY,"ytick.color":NAVY,
                     "axes.titlesize":9.5,"axes.titleweight":"bold"})

# a LADDER is a matched family: same recipe, same pooling, only size varies.
LADDERS = {
  "DINOv2 (self-supervised, cls+patch)":
      (["img_small","img_base","img_large","img_giant"], BLUE, "o"),
  "contrastive text embedders":
      (["txt_bge","txt_qwen06","txt_qwen4"], TEAL, "^"),
}

fig, ax = plt.subplots(figsize=(7.2, 4.6), dpi=200)
for title,(keys,col,mk) in LADDERS.items():
    pts=[(KNOWN_PARAMS[k], best_cross(k)[0], k) for k in keys
         if k in names and k in KNOWN_PARAMS]
    if len(pts) < 2: continue
    pts.sort()
    ax.plot([p for p,_,_ in pts], [r for _,r,_ in pts], mk+"-",
            color=col, ms=7, lw=1.8, label=title)
    for p,r,k in pts:
        ax.annotate(lab(k) + chr(10) + f"{r:.3f}", (p,r), fontsize=6.8,
                    xytext=(4,5), textcoords="offset points", color=col)

# encoders outside any matched ladder: plotted as unconnected reference marks
OTHER=[k for k in names if k in KNOWN_PARAMS
       and not any(k in v[0] for v in LADDERS.values())]
if OTHER:
    ax.scatter([KNOWN_PARAMS[k] for k in OTHER],
               [best_cross(k)[0] for k in OTHER],
               c=GREY, marker="s", s=26, zorder=2, label="not in a matched ladder")
    for k in OTHER:
        ax.annotate(lab(k), (KNOWN_PARAMS[k], best_cross(k)[0]), fontsize=6.2,
                    xytext=(4,-9), textcoords="offset points", color=GREY)

ax.set_xscale("log")
ax.set_xlabel("parameters (billions, log)")
ax.set_ylabel("best cross-modal rho")
ax.set_title("Does cross-modal agreement strengthen with capacity?" + chr(10) +
             "raw geometry, no map fitted, all %d items" % N)
ax.grid(alpha=0.25, lw=0.5)
ax.legend(frameon=False, fontsize=7.5, loc="lower left")
fig.tight_layout()
fig.savefig(os.path.join(OUT, "scale_ladder_rho.png"), bbox_inches="tight")
plt.show(); plt.close(fig)
print("saved scale_ladder_rho.png")
print("")
print("Each connected line is a MATCHED ladder: one training recipe, one")
print("pooling, capacity the only variable. Grey squares share no recipe with")
print("anything and are reference marks, not ladder points - a line through")
print("them would imply a comparison that was never controlled.")

In [ ]:
# Cell 10 — score the results against the pre-registration from Cell 2
#
# The predictions were written before any number existed. This cell reports
# CONFIRMED / FALSIFIED against them, either way, and records the verdict to disk.

pre = json.load(open(PREREG_PATH))
verdict = {}
print("=" * 72)
print("VERDICT AGAINST PRE-REGISTRATION")
print("=" * 72)

# P1 — image transfer
if "img_giant" in transfer:
    t = transfer["img_giant"]
    got, pw, rpd = t["pct_of_native"], t["powered"], t["rows_per_dim"]
    ok, marg = got >= 92.9, got >= 90.9
    verdict["P1_image_transfer"] = dict(value=got, confirmed=bool(ok and pw),
                                        powered=bool(pw), rows_per_dim=rpd)
    print(f"\nP1  DINOv2-giant transfer >= 92.9% (DINOv2-large)")
    print(f"    measured {got:.1f}% at {rpd:.1f} rows/dim")
    if not pw:
        print("    -> INCONCLUSIVE. The fit is underpowered, so a low value here")
        print("       carries no information about capacity. Re-run with cls pooling")
        print("       (1536d, 5.6 rows/dim) before drawing any conclusion.")
    else:
        print(f"    -> {'CONFIRMED' if ok else ('WITHIN NOISE' if marg else 'FALSIFIED')}")
        if not ok:
            print("    NOTE: a genuinely lower value from a POWERED fit is the more")
            print("    interesting result - it bounds the capacity ladder. Report as measured.")

# P2 — image rho
if "img_giant" in names:
    got, partner = best_cross("img_giant")
    ok = got >= 0.418
    verdict["P2_image_rho"] = dict(value=got, partner=partner, confirmed=bool(ok))
    print(f"\nP2  DINOv2-giant best cross-modal rho >= 0.418 (DINOv2-large)")
    print(f"    measured {got:.3f} with {lab(partner)}  ->  {'CONFIRMED' if ok else 'FALSIFIED'}")

# P3 — the scale caveat itself
bge_best = best_cross("txt_bge")[0] if "txt_bge" in names else 0.320
for tk, cfg in TEXT_MODELS.items():
    if tk not in names: continue
    got, partner = best_cross(tk)
    is_scale = "scale" in cfg.get("role", "")
    key = "P3_text_rho_SCALE" if is_scale else "P3b_text_rho_CONTROL"
    ok = got >= bge_best
    verdict[f"{key}_{tk}"] = dict(value=got, partner=partner, vs_bge=bge_best,
                                  confirmed=bool(ok), role=cfg.get("role","-"))
    print("")
    if is_scale:
        print(f"P3  SCALE TEST - {lab(tk)} ({cfg['params_b']}B, "
              f"{cfg['params_b']/0.568:.1f}x bge-m3)")
        print(f"    best cross-modal rho {got:.3f} with {lab(partner)}, "
              f"vs bge-m3's {bge_best:.3f}")
        print(f"    ->  {'CONFIRMED' if ok else 'FALSIFIED'}")
        print("    This is the direct test of the scale caveat (Defense_Brief section 8).")
        print("    rho fits nothing, so this model's width carries no statistical penalty.")
    else:
        print(f"P3b ARCHITECTURE CONTROL - {lab(tk)} ({cfg['params_b']}B, "
              f"{cfg['params_b']/0.568:.2f}x bge-m3 - SAME scale)")
        print(f"    best cross-modal rho {got:.3f} with {lab(partner)}, "
              f"vs bge-m3's {bge_best:.3f}")
        print("    A difference here is architecture or training data, NOT capacity.")

# P4 — decoupling (E.12 replication)
if "img_giant" in transfer and "img_giant" in names:
    d_rho  = best_cross("img_giant")[0] - 0.418
    d_pct  = transfer["img_giant"]["pct_of_native"] - 92.9
    same   = (d_rho >= 0) == (d_pct >= 0)
    verdict["P4_decoupling"] = dict(d_rho=d_rho, d_pct=d_pct, moved_together=bool(same))
    print(f"\nP4  do rho and transfer move together?")
    print(f"    d_rho {d_rho:+.3f}, d_transfer {d_pct:+.1f} pts  ->  "
          f"{'same direction' if same else 'OPPOSITE - replicates E.12'}")

json.dump({k: {kk: (float(vv) if isinstance(vv,(int,float,np.floating)) else vv)
               for kk, vv in v.items()} for k, v in verdict.items()},
          open(os.path.join(OUT, "verdict_scale_extension.json"), "w"), indent=2)

print("\n" + "=" * 72)
print("All artefacts in:", OUT)
print("  prereg_scale_extension.json   - written before measuring")
print("  rho_matrix_extended.npz       - full pairwise rho, all encoders")
print("  transfer_extended.json        - R@1 transfer / native / control")
print("  scale_ladder.png              - the two-panel capacity figure")
print("  verdict_scale_extension.json  - scored against the pre-registration")
print("\nSend the PNG and the two JSONs to Claude to fold into the documents.")